# Corpus-regression GRPO

In [ ]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.corpus_regression.config import data_dir
from src.experiments.corpus_regression.grpo import CorpusRegressionGRPOConfig
from src.experiments.corpus_regression.utils import dim_averaged_metrics_from_parquet

repo_root = get_repo_base()
device = torch.device("cuda:0")

### Configs

In [ ]:
config = CorpusRegressionGRPOConfig.get_canonical(
    dataset_base_folder=data_dir(),
    study_base_folder=repo_root / "artifacts" / "corpus-regression-grpo-example",
    num_lookforward_tokens=4,
    train_epochs=2,
    num_rollouts_per_sample=128,
    gaussian_stdev=1.0,
)

display(config.visualize())

In [3]:
state = config.initialize(device=device)
state.run_training()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

grpo epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

grpo epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

### Results

In [4]:
metrics = pl.read_parquet(config.study_folder / "metrics.parquet")
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,val_target_xx,val_target_xy,val_target_yy,val_target_n
i64,list[f64],list[f64],list[f64],f64,list[f64],list[f64],list[f64],f64
0,"[3786.365967, 4624.655762, … 3182.299072]","[424.136475, 514.926636, … 133.327881]","[49984.0, 49984.0, … 49984.0]",49984.0,"[1110.559692, 1178.001221, … 1897.81189]","[705.839783, 796.674805, … 501.85968]","[49920.0, 49920.0, … 49920.0]",49920.0
1,"[1655.671265, 1929.874634, … 1370.344116]","[512.99884, 666.07428, … 310.011414]","[49984.0, 49984.0, … 49984.0]",49984.0,"[2883.271973, 2176.820801, … 360.385559]","[1169.598877, 1095.154053, … 63.772282]","[49920.0, 49920.0, … 49920.0]",49920.0


In [5]:
dim_averaged_metrics_from_parquet(metrics, split="train").join(
    dim_averaged_metrics_from_parquet(metrics, split="val"), on="epoch"
)

epoch,train_avg_corr,train_avg_rsq,val_avg_corr,val_avg_rsq
i64,f64,f64,f64,f64
0,0.034693,-0.057329,0.06278,-0.006839
1,0.06635,-0.008915,0.061602,-0.010331


In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_df = pl.read_parquet(
    config.study_folder / str(last_epoch) / "validation.parquet"
)
validation_df.head()

model_preds,target
list[f64],list[f64]
"[-0.251953, 0.24707, … -0.053467]","[-1.0, 1.0, … -1.0]"
"[-0.255859, 0.261719, … -0.060791]","[-1.0, 1.0, … 1.0]"
"[-0.188477, 0.061523, … 0.090332]","[-1.0, 1.0, … -1.0]"
"[-0.251953, 0.15332, … 0.02771]","[1.0, -1.0, … 1.0]"
"[-0.211914, 0.296875, … -0.09668]","[-1.0, 1.0, … -1.0]"


: 